In [16]:
import os
print(os.getcwd())
print(os.listdir("../data"))

d:\ML Project - E-commerce product search with semantic ranking\notebooks
['corpus.parquet', 'esci_us_small.parquet', 'qrels.parquet', 'queries.parquet', 'shopping_queries_dataset_examples.parquet', 'shopping_queries_dataset_products.parquet']


In [17]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

DATA = Path(r"d:\ML Project - E-commerce product search with semantic ranking\data")

examples = pd.read_parquet(DATA / "shopping_queries_dataset_examples.parquet")
products_raw = pd.read_parquet(DATA / "shopping_queries_dataset_products.parquet")

print("Examples:", examples.shape)
print("Products:", products_raw.shape)

Examples: (2621288, 9)
Products: (1814924, 7)


In [18]:
examples_us = examples[
    (examples["product_locale"] == "us") &
    (examples["small_version"] == 1)
]
products_us = products_raw[products_raw["product_locale"] == "us"]

df = examples_us.merge(products_us, on=["product_id", "product_locale"], how="left")

print("Rows:", df.shape)
print("Unique queries:", df["query_id"].nunique())
print("Unique products:", df["product_id"].nunique())

df.to_parquet(DATA / "esci_us_small.parquet", index=False)
print("Saved!")

Rows: (601354, 14)
Unique queries: 29844
Unique products: 482105
Saved!


In [19]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = re.sub(r"<[^>]+>", " ", str(text))
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [20]:
products = (
    df[["product_id", "product_title", "product_description",
        "product_bullet_point", "product_brand", "product_color"]]
    .drop_duplicates(subset="product_id")
    .copy()
)
print("Unique products:", len(products))

Unique products: 482105


In [21]:
cols = ["product_title", "product_description",
        "product_bullet_point", "product_brand", "product_color"]

for col in cols:
    products[col] = products[col].apply(clean_text)

products = products[products["product_title"] != ""]
print("After cleaning:", len(products))

After cleaning: 482105


In [22]:
products["search_text"] = (
    products["product_title"] + ". "
    + products["product_brand"] + ". "
    + products["product_color"] + ". "
    + products["product_bullet_point"].str[:300] + " "
    + products["product_description"].str[:200]
).str.replace(r"\s+", " ", regex=True).str.strip()

products["search_text"].iloc[0]

'RamPro 10" All Purpose Utility Air Tires/Wheels with a 5/8" Diameter Hole with Double Sealed Bearings (Pack of 2). RamPro. 10 Inch. ✓ The Ram-Pro Ten Inch ready to install Air Tire Wheels will be the best replacement for your hand truck (or anything similar) tires and wheels with easy installation and great performance. ✓ SPECS: The 2 Ram-Pro Air Tires sizes are: High: 10" - Wide: 3" - Hole Diameter: 5/8" - Hub Depth: 1-3/4" wit About The Ram-Pro All Purpose Utility 10" Air Tires Wheels Pack of 2. The Problem: Your handtrucks tires are flat but why should you spend so much money to buy a compete new handtruck? Our Solution: T'

In [23]:
test_queries = df[df["split"] == "test"][["query_id", "query"]].drop_duplicates()
print("Available test queries:", len(test_queries))

eval_queries = test_queries.sample(n=500, random_state=42).reset_index(drop=True)
eval_queries.head()

Available test queries: 8956


,query_id,query
0,53255,inflatable dildo
1,42158,forget me not artificial flowers
2,47203,gray overalls men
3,56046,jms580
4,87048,remove stain from bath tub


In [24]:
gain_map = {"E": 3, "S": 2, "C": 1, "I": 0}

qrels = df[df["query_id"].isin(eval_queries["query_id"])][
    ["query_id", "product_id", "esci_label"]
].copy()
qrels["gain"] = qrels["esci_label"].map(gain_map)

print("Judgments:", len(qrels))
qrels["esci_label"].value_counts()

Judgments: 10125


esci_label
E    4261
S    3821
I    1684
C     359
Name: count, dtype: int64

In [25]:
eval_product_ids = set(qrels["product_id"])

others = products[~products["product_id"].isin(eval_product_ids)]
distractors = others.sample(n=min(30000, len(others)), random_state=42)

corpus = pd.concat([
    products[products["product_id"].isin(eval_product_ids)],
    distractors
]).drop_duplicates("product_id").reset_index(drop=True)

print("Corpus size:", len(corpus))
print("Eval queries:", len(eval_queries))
print("Judgments:", len(qrels))

Corpus size: 40063
Eval queries: 500
Judgments: 10125


In [26]:
corpus.to_parquet(DATA / "corpus.parquet", index=False)
eval_queries.to_parquet(DATA / "queries.parquet", index=False)
qrels.to_parquet(DATA / "qrels.parquet", index=False)
print("Saved!")

Saved!
